# 01B — Common canonical adapter

**Outcome:** convert any valid sector Pack into the same episode-aware
`SPEC-CORE`, with `SPLITS` and a physically separate `SPEC-EVAL`.

There is no ONT, splitter, well, valve or native metric logic anywhere in
this notebook. Sector notebooks own translation; this one selects a completed
Pack, calls the shared adapter, and proves the contract holds.

## 1. Setup and sector switch

In [ ]:
import os
import sys
import tempfile
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

try:
    import duckdb
except ImportError:
    get_ipython().system("pip install -q duckdb")
    import duckdb

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

default_data_root = (Path("/content/drive/MyDrive/anomaly_detection")
                     if IN_COLAB else Path.home() / "anomaly_detection_data")
DATA_ROOT = Path(os.getenv("ANOMALY_DATA_ROOT")
                 or os.getenv("ANOMALY_DRIVE_ROOT")
                 or default_data_root).expanduser()
default_code_root = (DATA_ROOT / "research" / "milestone1" if IN_COLAB
                     else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
                     else Path.cwd() / "notebooks" / "drive_research")
NOTEBOOK_HOME = Path(os.getenv("ANOMALY_NOTEBOOK_HOME", default_code_root)).expanduser()
if not (NOTEBOOK_HOME / "milestone1_core.py").is_file():
    raise FileNotFoundError(f"milestone1_core.py was not found in {NOTEBOOK_HOME}")
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import (
    CORE_SCHEMAS,
    CORE_VERSION,
    GAP_TOLERANCE_FACTOR,
    build_canonical,
    check_core,
    check_evaluation,
    core_fingerprint,
    read_json,
    save_pack,
)

SECTOR = os.getenv("ADAPTER_SECTOR", "telecom")           # "telecom" or "petrobras_3w"
BUILD_CANONICAL = os.getenv("BUILD_CANONICAL", "1") == "1"
RUN_CONTRACT_TESTS = os.getenv("RUN_CONTRACT_TESTS", "1") == "1"
RUN_FULL_CORE_AUDIT = os.getenv("RUN_FULL_CORE_AUDIT", "0") == "1"
AS_OF_TS = os.getenv("CANONICAL_AS_OF_TS") or None

PACK_RUN_IDS = {"telecom": "telecom_pack_v0_7_1", "petrobras_3w": "real_wells_expanded_v0_7_2"}
CANONICAL_RUN_IDS = {"telecom": "telecom_core_v0_10_1_run1",
                     "petrobras_3w": "petrobras_3w_core_v0_10_1_run2"}
if SECTOR not in PACK_RUN_IDS:
    raise ValueError(f"Choose one of {list(PACK_RUN_IDS)}")

PACK_ROOT = DATA_ROOT / "outputs" / "packs" / SECTOR / os.getenv(
    "ADAPTER_PACK_RUN_ID", PACK_RUN_IDS[SECTOR])
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / os.getenv(
    "CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])

display(pd.Series({
    "runtime": "Colab + Drive" if IN_COLAB else "local Python",
    "data_root": str(DATA_ROOT),
    "code_root": str(NOTEBOOK_HOME),
    "sector": SECTOR,
    "pack_root": str(PACK_ROOT),
    "canonical_root": str(RUN_ROOT),
    "as_of_ts": AS_OF_TS or "all observations",
    "gap_tolerance_factor": GAP_TOLERANCE_FACTOR,
    "full_output_reread": RUN_FULL_CORE_AUDIT,
}, name="value").to_frame())

## 2. Materialise and validate once

`build_canonical()` validates the Pack, writes telemetry in bounded batches,
and derives bounds, gaps and exact duplicate counts with a spillable SQL
scan. Its bounded build checks are the default. Set `RUN_FULL_CORE_AUDIT=1`
only when you intentionally want to re-read and re-hash the complete output;
that independent audit is expensive and is not required after every build.

In [ ]:
display(pd.DataFrame([
    {"table": name, "columns": ", ".join(columns)} for name, columns in CORE_SCHEMAS.items()
]))

if BUILD_CANONICAL:
    started = time.perf_counter()
    run_manifest = build_canonical(PACK_ROOT, RUN_ROOT, include_evaluation=True, as_of_ts=AS_OF_TS)
    elapsed = time.perf_counter() - started
else:
    run_manifest = read_json(RUN_ROOT / "run_manifest.json")
    elapsed = float("nan")

if RUN_FULL_CORE_AUDIT:
    core_audit = check_core(RUN_ROOT / "SPEC-CORE")
    core_audit["audit_mode"] = "independent_full_reread"
else:
    core = run_manifest["core"]
    core_audit = {
        "audit_mode": "bounded_build_checks" if BUILD_CANONICAL else "manifest_read",
        "telemetry_rows": core["row_counts"]["telemetry"],
        "quality_counts": core["quality_counts"],
        "duplicate_keys": 0,
        "foreign_key_failures": 0,
        "fingerprint_verified": "during_materialisation",
        **{name: core["row_counts"][name] for name in list(CORE_SCHEMAS)[1:]},
    }
display(pd.Series(read_json(PACK_ROOT / "pack_manifest.json"), name="value").to_frame())
display(pd.Series(core_audit, name="value").to_frame())
print(f"build_canonical: {elapsed:.1f}s")

## 3. Contract fixtures

Small generic fixtures, not another scan of the sector Pack. Together they
prove:

- mounting evaluation cannot change `SPEC-CORE`;
- a deliberately leaky scorer fails when `SPEC-EVAL` is absent;
- metrics on different cadences keep independent timestamp grids;
- jitter is not a gap, but a missing scheduled observation is;
- a `recording` metric gets gaps **inside** an episode and never between two;
- an unavailable metric is absent, while a failed observation is `invalid`;
- observations after `as_of_ts` cannot change earlier canonical content;
- a duplicate key split across two Parquet parts is rejected.

The 01A original-versus-redacted test remains the primary translator-leakage
proof; these are deployment evidence for the shared code path.

In [ ]:
BASE = pd.Timestamp("2025-01-01", tz="UTC")


def observation(seconds, metric_id, value, *, episode="episode-1", quality=None):
    return {
        "event_ts": BASE + pd.Timedelta(seconds=seconds),
        "entity_id": "asset-1",
        "episode_id": episode,
        "metric_id": metric_id,
        "value": value,
        "quality_code": quality or ("invalid" if value is None else "measured"),
    }


def fixture_observations():
    rows = [observation(s, "fast_signal", float(s)) for s in range(21)]
    # 5 s metric: jitter at 5.2 s is tolerated, the 10 s sample is missing.
    rows += [observation(s, "slow_signal", v)
             for s, v in [(0, 20.0), (5.2, 20.5), (15, 21.0), (20, None)]]
    # Second recording of the same entity, 40 s later. No gap may be reported
    # between the two episodes, but the hole at 3 s inside it must be.
    rows += [observation(60 + s, "fast_signal", float(s), episode="episode-2")
             for s in (0, 1, 2, 5, 6)]
    return pd.DataFrame(rows)


def write_fixture_pack(destination, observations, *, evaluation=True, duplicate_part=False):
    metrics = sorted(observations["metric_id"].unique())
    episodes = observations[["episode_id", "entity_id"]].drop_duplicates()
    batches = [observations]
    if duplicate_part:
        batches.append(observations.head(1))
    save_pack(
        destination,
        sector="contract_test",
        pack_version="adapter-fixture-v1",
        source_info={"source_id": "contract-fixture", "files": []},
        telemetry=batches,
        catalogue=pd.DataFrame([
            (m, "test_asset", "gauge", "unit", "recording", 1.0 if m == "fast_signal" else 5.0)
            for m in metrics
        ], columns=CORE_SCHEMAS["metric_catalogue"]),
        entities=pd.DataFrame([("asset-1", "test_asset")], columns=["entity_id", "entity_type"]),
        episodes=episodes.assign(episode_basis="contract_fixture"),
        evaluation={"condition_states": pd.DataFrame([{
            "entity_id": "asset-1",
            "start_ts": observations["event_ts"].min(),
            "end_ts": observations["event_ts"].max(),
            "condition_code": "fixture_condition",
            "label_source": "contract_fixture",
            "source_instance_id": "episode-1",
        }])} if evaluation else {},
    )


def leaky_scorer(run_root):
    """A detector that cheats. It must fail when SPEC-EVAL is not mounted."""

    truth = Path(run_root) / "SPEC-EVAL"
    if not truth.is_dir():
        raise FileNotFoundError("SPEC-EVAL is not mounted")
    return sorted(path.name for path in truth.glob("*.parquet"))

In [ ]:
contract_status = "not_run"
if RUN_CONTRACT_TESTS:
    observations = fixture_observations()
    cutoff = BASE + pd.Timedelta(seconds=10)

    with tempfile.TemporaryDirectory() as temporary:
        temporary = Path(temporary)
        full = temporary / "pack_full"
        truncated = temporary / "pack_truncated"
        duplicated = temporary / "pack_duplicated"
        write_fixture_pack(full, observations)
        write_fixture_pack(truncated, observations.loc[observations["event_ts"].le(cutoff)],
                           evaluation=False)
        write_fixture_pack(duplicated, observations, evaluation=False, duplicate_part=True)

        nonfinite = observations.copy()
        nonfinite.loc[nonfinite.index[0], "value"] = float("inf")
        try:
            write_fixture_pack(temporary / "pack_nonfinite", nonfinite, evaluation=False)
        except ValueError as error:
            assert "non-finite values must be invalid" in str(error)
        else:
            raise AssertionError("A measured infinite value was accepted")

        mounted, unmounted = temporary / "mounted", temporary / "unmounted"
        build_canonical(full, mounted, include_evaluation=True)
        build_canonical(full, unmounted, include_evaluation=False)
        assert core_fingerprint(mounted / "SPEC-CORE") == core_fingerprint(unmounted / "SPEC-CORE")

        leaky_scorer(mounted)
        try:
            leaky_scorer(unmounted)
        except FileNotFoundError:
            pass
        else:
            raise AssertionError("Negative control read truth that was not mounted")

        telemetry = pd.concat(
            [pd.read_parquet(p) for p in sorted((mounted / "SPEC-CORE" / "telemetry").glob("*.parquet"))],
            ignore_index=True,
        )
        gaps = pd.read_parquet(mounted / "SPEC-CORE" / "collection_gaps.parquet")

        slow = telemetry.loc[telemetry["metric_id"].eq("slow_signal")]
        assert len(slow) == 4 and slow.sort_values("event_ts").iloc[-1]["quality_code"] == "invalid"
        assert "slow_signal" not in set(
            telemetry.loc[telemetry["episode_id"].eq("episode-2"), "metric_id"]
        ), "An unavailable metric must be absent, not a run of invalid rows"

        gap_keys = set(zip(gaps["episode_id"], gaps["metric_id"]))
        assert ("episode-1", "slow_signal") in gap_keys, "Missing scheduled observation must be a gap"
        assert ("episode-2", "fast_signal") in gap_keys, "A hole inside a recording must be a gap"
        assert len(gaps) == 2, f"Jitter or an inter-episode interval was called a gap: {gaps}"
        slow_gap = gaps.loc[gaps["metric_id"].eq("slow_signal")].iloc[0]
        assert slow_gap["gap_start"] == BASE + pd.Timedelta(seconds=10.2)
        assert slow_gap["gap_end"] == BASE + pd.Timedelta(seconds=15)

        full_as_of, truncated_as_of = temporary / "as_of_full", temporary / "as_of_truncated"
        build_canonical(full, full_as_of, include_evaluation=False, as_of_ts=cutoff)
        build_canonical(truncated, truncated_as_of, include_evaluation=False, as_of_ts=cutoff)
        assert core_fingerprint(full_as_of / "SPEC-CORE") == core_fingerprint(
            truncated_as_of / "SPEC-CORE")

        try:
            build_canonical(duplicated, temporary / "duplicate_run", include_evaluation=False)
        except ValueError as error:
            assert "across Pack parts" in str(error)
        else:
            raise AssertionError("A duplicate key across Parquet parts was accepted")

    contract_status = "pass"
    for message in [
        "truth mounting cannot change SPEC-CORE",
        "the negative control fails without SPEC-EVAL",
        "mixed cadence, jitter and missing observations stay distinct",
        "gaps are found inside a recording and never between recordings",
        "an unavailable metric is absent, not invalid",
        "future observations cannot change an earlier as-of run",
        "duplicate keys across Parquet parts are rejected",
        "non-finite values cannot be marked as measured",
    ]:
        print(f"PASS — {message}")
else:
    print("NOT RUN — set RUN_CONTRACT_TESTS=1")

## 4. Evaluation integrity — reported, never mounted into the detector path

`check_evaluation()` reads `SPEC-EVAL` and `SPEC-CORE` together, which is why
it lives outside `check_core()` and is never called by a detector.

It answers one question the harness needs before it can be trusted: how many
labelled faults are actually **scoreable**? A fault on an entity that is not
in the registry, or in a window with no telemetry, is silently dropped by an
alert-to-fault matcher — and a silently dropped fault inflates precision.

In [ ]:
integrity = check_evaluation(RUN_ROOT)
display(pd.Series(integrity, name="value").to_frame())

if integrity.get("intervals_on_unknown_entities"):
    print("WARNING — evaluation truth references entities absent from SPEC-CORE")
if integrity.get("intervals_outside_observed_window"):
    print("WARNING — some fault intervals lie outside any observed telemetry window")

## 5. Acceptance and output inspection

There is no hard-coded "sector-neutral" flag. Sector neutrality is shown by
the same `build_canonical()` call accepting either Pack unchanged.

In [ ]:
display(pd.Series({
    "contract_version": CORE_VERSION,
    "sector": SECTOR,
    "common_adapter_function": "build_canonical",
    "fingerprint_verified": core_audit["fingerprint_verified"],
    "global_key_audit": "pass",
    "contract_fixtures": contract_status,
    "evaluation_mounted": run_manifest["evaluation_mounted"],
    "core_tables": list(CORE_SCHEMAS),
}, name="result").to_frame())

for name in list(CORE_SCHEMAS)[1:]:
    frame = pd.read_parquet(RUN_ROOT / "SPEC-CORE" / f"{name}.parquet")
    print(f"\n{name}: {len(frame):,} rows")
    display(frame.head(8))

parts = sorted((RUN_ROOT / "SPEC-CORE" / "telemetry").glob("part-*.parquet"))
print(f"\ntelemetry: {len(parts)} parts, {run_manifest['core']['row_counts']['telemetry']:,} rows")
display(pd.read_parquet(parts[0]).head(8))

for name in run_manifest["split_rows"]:
    frame = pd.read_parquet(RUN_ROOT / "SPLITS" / f"{name}.parquet")
    print(f"\nSPLITS/{name}: {len(frame):,} rows (orchestration metadata, not model input)")
    display(frame.head(8))

print("\nSPEC-EVAL tables:", list(run_manifest["evaluation_rows"]))
print("Canonical run:", RUN_ROOT)
print("Next: 02_CANONICAL_EDA.ipynb")